# 10 · LGWR / GWR

Template para el modelo geoestadístico local. En el repo actual la implementación disponible es `GWRModel`, así que este notebook queda preparado para usar ese wrapper y analizar coeficientes locales.

## Hipótesis del modelo

- La relación entre features y `precio_m2` cambia según la localización.
- El modelo debería capturar heterogeneidad espacial mejor que una regresión global.
- La interpretabilidad principal viene por mapas de coeficientes locales y diagnóstico de residuos.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ml_core.models.gwrmodel import GWRModel
from ml_core.evaluation.modelEvaluator import regression_metrics
from ml_core.visualization.mapper import *
from ml_core import load_model_config, save_model_config

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "10_lgwr"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


## Datos y configuración

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "splits"

subset_config = {
    "train_n": 2500,
    "val_n": 800,
    "test_n": 800,
    "random_state": 42,
}


def subset_split(gdf, n_rows, random_state):
    if n_rows is None or len(gdf) <= n_rows:
        return gdf.copy().reset_index(drop=True)
    return (
        gdf.sample(n=n_rows, random_state=random_state)
        .sort_index()
        .reset_index(drop=True)
    )


train_raw = pd.read_csv(DATA_PATH / "arg_venta_data_train.csv")
gdf_train_full = gpd.GeoDataFrame(
    train_raw,
    geometry=gpd.points_from_xy(
        train_raw["longitud"],
        train_raw["latitud"]
    ),
    crs="EPSG:4326"
)

test_raw = pd.read_csv(DATA_PATH / "arg_venta_data_test.csv")
gdf_test_full = gpd.GeoDataFrame(
    test_raw,
    geometry=gpd.points_from_xy(
        test_raw["longitud"],
        test_raw["latitud"]
    ),
    crs="EPSG:4326"
)

val_raw = pd.read_csv(DATA_PATH / "arg_venta_data_val.csv")
gdf_val_full = gpd.GeoDataFrame(
    val_raw,
    geometry=gpd.points_from_xy(
        val_raw["longitud"],
        val_raw["latitud"]
    ),
    crs="EPSG:4326"
)

gdf_train = subset_split(gdf_train_full, subset_config["train_n"], subset_config["random_state"])
gdf_val = subset_split(gdf_val_full, subset_config["val_n"], subset_config["random_state"] + 1)
gdf_test = subset_split(gdf_test_full, subset_config["test_n"], subset_config["random_state"] + 2)

target_col = "log_precio"
coord_cols = ["longitud", "latitud"]
feature_cols = [
    "area_m2_total",
    "area_m2_descubierta",
    "ambientes",
    "antiguedad",
    "expensas",
    "estado_num",
]

print(
    f"Subset LGWR -> train={len(gdf_train)} / val={len(gdf_val)} / test={len(gdf_test)}"
)


Subset LGWR -> train=2500 / val=800 / test=800


In [3]:
X_train = gdf_train[feature_cols]
y_train = gdf_train[target_col]
coords_train = gdf_train[coord_cols].to_numpy()

X_test = gdf_test[feature_cols]
y_test = gdf_test[target_col]
coords_test = gdf_test[coord_cols].to_numpy()

X_val = gdf_val[feature_cols]
y_val = gdf_val[target_col]
coords_val = gdf_val[coord_cols].to_numpy()


## Entrenamiento

In [4]:
gwr_params = {
    "kernel": "bisquare",
    "fixed": False,
}

gwr_params


{'kernel': 'bisquare', 'fixed': False}

## Tuning

Podés barrer `kernel`, `fixed`, `bw` y distintas definiciones de features. Si hacés tuning, dejá el criterio registrado en `run_config.json`.

In [5]:
config_path = PROJECT_ROOT / "notebooks" / "cache" / "lgwr_best_config.json"
saved_config = load_model_config(config_path)

if saved_config is None:
    tuning_model = GWRModel(gwr_params=gwr_params.copy())
    tuning_model.tune_hyperparameters(
        X_train,
        y_train.values.reshape(-1, 1),
        coords_train,
    )

    save_model_config(
        tuning_model,
        config_path,
        extra={
            "target": target_col,
            "features": feature_cols,
            "subset_config": subset_config,
        },
    )

    saved_config = load_model_config(config_path)

gwr_params = saved_config.get("gwr_params", gwr_params)
selected_bw = saved_config.get("selected_bw", None)

best_config = {
    "gwr_params": gwr_params,
    "selected_bw": selected_bw,
}
best_config


/home/saneliges/Escritorio/PredictorPrecioMetroCuadradoAlquileres/.venv/lib/python3.12/site-packages/spglm/iwls.py:37: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.5587156349666885e-17.
  xtx_inv_xt = linalg.solve(xtx, xT)
/home/saneliges/Escritorio/PredictorPrecioMetroCuadradoAlquileres/.venv/lib/python3.12/site-packages/spglm/iwls.py:37: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.4094877352205278e-17.
  xtx_inv_xt = linalg.solve(xtx, xT)


LinAlgError: A singular matrix detected: slice(s) [0] are singular.

## Evaluación global

In [ ]:
gwr_params = best_config["gwr_params"]
selected_bw = best_config["selected_bw"]

model = GWRModel(gwr_params=gwr_params)
fit_kwargs = {}
if selected_bw is not None:
    fit_kwargs["bw"] = selected_bw

model.fit(X_train, y_train, coords_train, **fit_kwargs)
model

y_pred_log = model.predict(
    X_val,
    coords_val
)

# revertir log
y_pred = np.exp(np.asarray(y_pred_log).reshape(-1))
y_true = np.exp(np.asarray(y_val).reshape(-1))

metrics = regression_metrics(
    y_true,
    y_pred
)

metrics


## Diagnóstico espacial

In [ ]:
barrios_path = PROJECT_ROOT / 'GeoData' / 'barrios.geojson'

df_grid, barrios, std = generar_grid_predicciones(
    model,
    gdf_val,
    feature_cols
)

mapa = MapaPrecio(df_grid, barrios)

mapa.plot()

#mapa.save("mapa_modelo_lgwr.png")

#mapa.save("mapa_modelo_lgwr.pdf")



## Interpretación

En GWR la interpretación más útil no suele ser una `feature importance` global, sino:

- coeficientes locales
- estabilidad espacial de cada feature
- signos dominantes por zona
- residuos espacialmente agrupados

In [ ]:
barrios = gpd.read_file(barrios_path)
barrios = barrios.to_crs(gdf_train.crs)
surfaces = model.plot_gwr_surfaces(
    gdf=gdf_train,
    feature_names= feature_cols,
    barrios = barrios.to_crs(gdf_train.crs),
    grid_size=300,
    return_surfaces=True
)


## Export de artefactos

In [ ]:
# test_export = gdf_test[[target_col] + coord_cols].copy()
# test_export = test_export.rename(columns={target_col: "y_true"})
# test_export["y_pred"] = np.asarray(y_pred).reshape(-1)
# test_export["residual"] = test_export["y_true"] - test_export["y_pred"]
# test_export["split"] = "test"
# test_export.to_parquet(OUTPUT_DIR / "test_predictions.parquet", index=False)
# local_betas.to_parquet(OUTPUT_DIR / "interpretability.parquet", index=False)
# (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
# (OUTPUT_DIR / "run_config.json").write_text(json.dumps(best_config, indent=2, ensure_ascii=False))